### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==5.3.0
!pip install --no-deps trl==0.22.2

### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-9B",
    max_seq_length = 16384,
    load_in_4bit = False,
    load_in_8bit = False,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.11: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Add LoRA adapters

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "out_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 8875,
    use_rslora = False,  # rank stabilized LoRA
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


## Dataset prep

In [5]:
from datasets import load_dataset

# Path to JSONL SFT dataset
DATA_PATH = "scad_sft_dataset.jsonl"   # change if needed

# Load local JSONL
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

# Basic sanity check
required_cols = {"prompt", "response"}
missing = required_cols - set(dataset.column_names)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found: {dataset.column_names}")

# Convert prompt/response -> Qwen chat conversations
def generate_conversation(examples):
    prompts = examples["prompt"]
    responses = examples["response"]
    conversations = []
    for prompt, response in zip(prompts, responses):
        conversations.append([
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response},
        ])
    return {"conversations": conversations}

dataset = dataset.map(generate_conversation, batched=True)

# Apply Qwen chat template and save final training text
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
dataset = dataset.remove_columns(["prompt", "response", "conversations"])

print(dataset)
print(dataset[0]["text"][:1000])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/139 [00:00<?, ? examples/s]

Map:   0%|          | 0/139 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 139
})
<|im_start|>user
Task: Modify the OpenSCAD program according to the instruction.

Return the full updated OpenSCAD code.
Do not explain anything.

Instruction:
Add one cube at an absolute world coordinate. Keep all existing geometry unchanged. The new cube must use corner placement (center=false), have size [4,5,6], and corner origin exactly at [0,0,0].

Existing OpenSCAD:
```scad
union() {
  translate([10, 0, 0]) sphere(r=1.2);
}
```<|im_end|>
<|im_start|>assistant
<think>

</think>

union() {
  translate([10, 0, 0]) sphere(r=1.2);
  translate([0, 0, 0]) cube(size=[4,5,6], center=false);
}<|im_end|>



Let's see how the chat template did

In [6]:
dataset[100]['text']

'<|im_start|>user\nTask: Modify the OpenSCAD program according to the instruction.\n\nReturn the full updated OpenSCAD code.\nDo not explain anything.\n\nInstruction:\nComplete the arrangement: this should be a 4 by 4 checkerboard pattern of cubes keeping only the even-parity cells where row+column is even. Each cube has size [1.5,1.5,1.5], uses corner placement, and grid origins are multiples of 3 in x and y starting at [0,0,0]. The cube at [3,3,0] is missing from the parity pattern. Add the missing cube.\n\nExisting OpenSCAD:\n```scad\nunion() {\n\ttranslate([0, 0, 0]) cube(size=[1.5,1.5,1.5], center=false);\n\ttranslate([6, 0, 0]) cube(size=[1.5,1.5,1.5], center=false);\n\ttranslate([9, 3, 0]) cube(size=[1.5,1.5,1.5], center=false);\n\ttranslate([0, 6, 0]) cube(size=[1.5,1.5,1.5], center=false);\n\ttranslate([6, 6, 0]) cube(size=[1.5,1.5,1.5], center=false);\n\ttranslate([3, 9, 0]) cube(size=[1.5,1.5,1.5], center=false);\n\ttranslate([9, 9, 0]) cube(size=[1.5,1.5,1.5], center=false)

### Train the model

In [7]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        dataset_num_proc = 1,
        per_device_train_batch_size = 4, # lower if OOM
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2,
        max_steps = -1,
        learning_rate = 5e-5,
        logging_steps = 3,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [8]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map (num_proc=16):   0%|          | 0/139 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/139 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/139 [00:00<?, ? examples/s]

In [9]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.494 GB.
17.676 GB of memory reserved.


To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [10]:
trainer_stats = trainer.train()
model.save_pretrained("qwen_lora_9b")
tokenizer.save_pretrained("qwen_lora_9b")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 139 | Num Epochs = 2 | Total steps = 36
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 32,243,712 of 9,442,057,456 (0.34% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
3,0.118404
6,0.100126
9,0.109831
12,0.053009
15,0.048846
18,0.034586
21,0.040506
24,0.046715
27,0.029060
30,0.030405


['qwen_lora_9b/processor_config.json']

In [11]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

203.1372 seconds used for training.
3.39 minutes used for training.
Peak reserved memory = 21.43 GB.
Peak reserved memory for training = 3.754 GB.
Peak reserved memory % of max memory = 54.261 %.
Peak reserved memory for training % of max memory = 9.505 %.


Zip and download in the side panel

In [12]:
!zip -r qwen_lora_9b.zip qwen_lora_9b

  adding: qwen_lora_9b/ (stored 0%)
  adding: qwen_lora_9b/processor_config.json (deflated 71%)
  adding: qwen_lora_9b/adapter_config.json (deflated 58%)
  adding: qwen_lora_9b/chat_template.jinja (deflated 77%)
  adding: qwen_lora_9b/adapter_model.safetensors (deflated 8%)
  adding: qwen_lora_9b/tokenizer.json (deflated 80%)
  adding: qwen_lora_9b/tokenizer_config.json (deflated 65%)
  adding: qwen_lora_9b/README.md (deflated 65%)


Export q4km GGUF


In [13]:
# Save LoRA (optional but recommended)
model.save_pretrained("qwen_lora_9b")
tokenizer.save_pretrained("qwen_lora_9b")

# Export GGUF (this already includes merging)
model.save_pretrained_gguf(
    "qwen_openscad_finetune_9b_gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 4 files from cache to `qwen_openscad_finetune_9b_gguf`:   0%|          | 0/4 [00:00<?, ?it/s]
Unsloth: Copying 4 files from cache to `qwen_openscad_finetune_9b_gguf`:  25%|██▌       | 1/4 [00:13<00:41, 13.84s/it]
Unsloth: Copying 4 files from cache to `qwen_openscad_finetune_9b_gguf`:  50%|█████     | 2/4 [00:30<00:31, 15.62s/it]
Unsloth: Copying 4 files from cache to `qwen_openscad_finetune_9b_gguf`:  75%|███████▌  | 3/4 [00:48<00:16, 16.53s/it]
Unsloth: Copying 4 files from cache to `qwen_openscad_finetune_9b_gguf`: 100%|██████████| 4/4 [01:01<00:00, 15.30s/it]


Successfully copied all 4 files from cache to `qwen_openscad_finetune_9b_gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 32201.95it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:13<00:00, 18.27s/it]


Unsloth: Merge process complete. Saved to `/content/qwen_openscad_finetune_9b_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen_openscad_finetune_9b_gguf_gguf/Qwen3.5-9B.BF16.gguf', 'qwen_openscad_finetune_9b_gguf_gguf/Qwen3.5-9B.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['qwen_openscad_finetune_9b_gguf_gguf/Qwen3.5-9B.Q4_K_M.gguf', 'qwen_openscad_finetune_9b_gguf_gguf/Qwen3.5-9B.BF16-mmproj.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/Qwen3.5-9B'. Skipping Ollama Modelfile


Unsloth: example usage for Multimodal LLMs: /root/.unsloth/llama.cpp/llama-mtmd-cli -m qwen_openscad_finetune_9b_gguf_gguf/Qwen3.5-9B.Q4_K_M.gguf --mmproj qwen_openscad_finetune_9b_gguf_gguf/Qwen3.5-9B.BF16-mmproj.gguf
Unsloth: load image inside llama.cpp runner: /image test_image.jpg
Unsloth: Prompt model to describe the image


{'save_directory': 'qwen_openscad_finetune_9b_gguf',
 'gguf_directory': 'qwen_openscad_finetune_9b_gguf_gguf',
 'gguf_files': ['qwen_openscad_finetune_9b_gguf_gguf/Qwen3.5-9B.Q4_K_M.gguf',
  'qwen_openscad_finetune_9b_gguf_gguf/Qwen3.5-9B.BF16-mmproj.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': True,
 'fix_bos_token': False}